<a href="https://colab.research.google.com/github/johanhoffman/DD2365_FEniCSx/blob/main/Poisson_equation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The Poisson equation**
**Johan Hoffman**

# **Abstract**

This short report shows an example of how to use FEniCSx to solve the Poisson equation,
which is used in the course DD2365 Advanced Computation in Fluid Mechanics at KTH Royal
Institute of Technology.

[DD2365 course website.](https://www.kth.se/social/course/DD2365/)

To run this code in Google Colab use the drop-down menu: Runtime / Run all

# **About the code**

In [ ]:
# Copyright (C) 2020-2026 Johan Hoffman (jhoffman@kth.se)
# SPDX-License-Identifier: MIT
#
# FEniCSx port of the DD2365 Poisson equation notebook.
# Original FEniCS version: https://github.com/johanhoffman/DD2365
#
# This template is maintained by Johan Hoffman.
# Please report problems to jhoffman@kth.se

# **Set up environment**

In [ ]:
# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
import numpy as np
from mpi4py import MPI
import ufl
from dolfinx import fem
from dolfinx.fem import functionspace, Function, assemble_scalar, form
from dolfinx.fem.petsc import NonlinearProblem
import matplotlib.pyplot as plt

In [ ]:
# --- canonical: gmsh_rect_minus_circles v1 ---
import numpy as np
import gmsh
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio


def gmsh_rect_minus_circles(L, H, circles, resolution):
    """Rectangle [0,L]×[0,H] minus circular holes, meshed with gmsh OCC.

    Parameters
    ----------
    L, H : float
        Rectangle dimensions.
    circles : list of (cx, cy, r)
        Circle centres and radii to subtract.
    resolution : int
        Global mesh size target: lc = 1/resolution.  mshr's ``resolution``
        parameter has slightly different semantics (CGAL internal); here lc
        sets a uniform global target — cell count will differ slightly.

    Returns
    -------
    msh : dolfinx.mesh.Mesh
    cell_tags : dolfinx.mesh.MeshTags   (fluid domain tag = 10)

    Note
    ----
    Facet tags are not produced here.  Call ``tag_boundaries(msh, L, H)``
    on the final mesh (after any refinement) to obtain boundary MeshTags.
    """
    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    rect = gmsh.model.occ.addRectangle(0.0, 0.0, 0.0, L, H)
    disks = [(2, gmsh.model.occ.addDisk(cx, cy, 0.0, r, r)) for cx, cy, r in circles]
    if disks:
        gmsh.model.occ.cut([(2, rect)], disks)
    gmsh.model.occ.synchronize()

    lc = 1.0 / resolution
    gmsh.option.setNumber("Mesh.MeshSizeMin", 0.5 * lc)
    gmsh.option.setNumber("Mesh.MeshSizeMax", lc)

    surfaces = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(2, [s[1] for s in surfaces], tag=10, name="domain")

    gmsh.model.mesh.generate(2)

    mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, rank=0, gdim=2)
    gmsh.finalize()
    return mesh_data.mesh, mesh_data.cell_tags
# --- end canonical: gmsh_rect_minus_circles ---

In [ ]:
# --- canonical: refine_cells v1 ---
import numpy as np
import dolfinx.mesh
from dolfinx.mesh import RefinementOption


def refine_cells(msh, predicate):
    """Refine all cells whose midpoint satisfies predicate (Plaza algorithm).

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    predicate : callable
        ``f(midpoints) -> bool array`` where ``midpoints`` has shape
        ``(ncells, gdim)``.

    Returns
    -------
    refined_msh : dolfinx.mesh.Mesh
        The refined mesh.
    parent_cells : np.ndarray[np.int32]
        For each refined cell: index of its parent cell in ``msh``.
        Shape ``(num_refined_cells,)``.
    parent_facets : np.ndarray[np.int8]
        For each refined cell: local index (0–3) of the parent facet it
        inherits, or -1 if interior.  Shape ``(num_refined_cells,)``.
        dolfinx 0.11: ``RefinementOption.parent_cell_and_facet``.
    """
    tdim = msh.topology.dim
    num_cells = msh.topology.index_map(tdim).size_local
    msh.topology.create_entities(1)
    msh.topology.create_connectivity(tdim, 0)
    midpoints = dolfinx.mesh.compute_midpoints(
        msh, tdim, np.arange(num_cells, dtype=np.int32)
    )
    marked = np.where(predicate(midpoints))[0].astype(np.int32)
    if len(marked) == 0:
        # No cells marked: return identity — parent_cells is trivial range,
        # parent_facets is all-(-1).
        n = msh.topology.index_map(tdim).size_local
        return msh, np.arange(n, dtype=np.int32), np.full(n, -1, dtype=np.int8)
    msh.topology.create_connectivity(tdim, 1)
    edges = dolfinx.mesh.compute_incident_entities(msh.topology, marked, tdim, 1)
    edges = np.unique(edges).astype(np.int32)
    return dolfinx.mesh.refine(
        msh, edges, option=RefinementOption.parent_cell_and_facet
    )
# --- end canonical: refine_cells ---

In [ ]:
# --- canonical: plot_helpers v1 ---
import numpy as np
import matplotlib.pyplot as plt


def plot_mesh(msh, title="Mesh"):
    """Plot a 2-D triangular mesh using matplotlib triplot."""
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def plot_p1(u, title="Solution"):
    """Plot a P1 scalar Function using matplotlib tripcolor (Gouraud shading)."""
    V = u.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]  # geometry connectivity (ncells, 3)
    ldm = V.dofmap.list             # DOF map (ncells, 3)
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = u.x.array.real[ldm.ravel()]
    fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
# --- end canonical: plot_helpers ---

In [ ]:
# --- canonical: export_xdmf v1 ---
import sys
import pathlib
from mpi4py import MPI
from dolfinx.io import XDMFFile


def export_xdmf(path, funcs, download=False):
    """Export a list of Functions to XDMF for ParaView.

    Parameters
    ----------
    path : str or Path
        Output .xdmf file path (an .h5 sidecar is written alongside it).
    funcs : list of dolfinx.fem.Function
        Functions to export (must share the same mesh).
    download : bool, optional
        If True *and* running on Colab, tar the .xdmf/.h5 pair and trigger
        a browser download.  Default False.
    """
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with XDMFFile(MPI.COMM_WORLD, str(path), "w") as xdmf:
        if funcs:
            xdmf.write_mesh(funcs[0].function_space.mesh)
        for f in funcs:
            xdmf.write_function(f)
    if download and "google.colab" in sys.modules:
        import tarfile
        from google.colab import files as colab_files
        tar_path = str(path.with_suffix(".tar.gz"))
        with tarfile.open(tar_path, "w:gz") as tar:
            tar.add(str(path), arcname=path.name)
            h5 = path.with_suffix(".h5")
            if h5.exists():
                tar.add(str(h5), arcname=h5.name)
        colab_files.download(tar_path)
# --- end canonical: export_xdmf ---

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]×[0,H] rectangle with holes.

    Assigns integer tags to all exterior boundary facets:

        left=1  (x ≈ 0),   right=2 (x ≈ L),
        lower=3 (y ≈ 0),   upper=4 (y ≈ H),
        objects=5 (remaining exterior facets — circle boundaries).

    Corners are unambiguous: boundary facets are edges, not vertices, so a
    corner vertex is shared by one vertical and one horizontal edge — each
    edge belongs to exactly one group and no facet is tagged twice.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    L, H : float
        Rectangle dimensions.
    eps : float, optional
        Coordinate tolerance.  Default ``1e-6 * max(L, H)``.

    Returns
    -------
    dolfinx.mesh.MeshTags
        Must be recomputed whenever the mesh changes (e.g. after refinement).
    """
    if eps is None:
        eps = 1e-6 * max(L, H)

    fdim = msh.topology.dim - 1
    msh.topology.create_entities(fdim)
    msh.topology.create_connectivity(fdim, msh.topology.dim)

    left  = locate_entities_boundary(msh, fdim, lambda x: x[0] <= eps)
    right = locate_entities_boundary(msh, fdim, lambda x: x[0] >= L - eps)
    lower = locate_entities_boundary(msh, fdim, lambda x: x[1] <= eps)
    upper = locate_entities_boundary(msh, fdim, lambda x: x[1] >= H - eps)

    known = np.unique(np.concatenate([left, right, lower, upper]))
    all_bdry = locate_entities_boundary(
        msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool)
    )
    objects = np.setdiff1d(all_bdry, known)

    indices = np.concatenate([left, right, lower, upper, objects]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(left),    1, dtype=np.int32),
        np.full(len(right),   2, dtype=np.int32),
        np.full(len(lower),   3, dtype=np.int32),
        np.full(len(upper),   4, dtype=np.int32),
        np.full(len(objects), 5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])
# --- end canonical: tag_boundaries ---

# **Introduction**

\
The Poisson equation takes the form

$-\Delta u = f,$

together with suitable boundary conditions.

Here we present a FEniCSx implementation of a finite element method to solve the Poisson
equation in 2D. The solution is visualized using matplotlib, and is also exported as
XDMF files which can be visualized in ParaView.

To derive the weak form of the equations, multiply the equation by $v\in V$, integrate
over the domain $\Omega$ and use Green's formula
$$(-\Delta u,v) = (\nabla u, \nabla v) - \langle\nabla u\cdot n, v\rangle_{\partial \Omega}$$

We seek a finite element approximation $u\in V$ such that

$$(\nabla u,\nabla v) - \langle\nabla u\cdot n, v\rangle_{\partial \Omega} = (f,v)$$

for all test functions $v \in V$.

$$(v,w) = \int_{\Omega} v\cdot w ~dx, \quad \langle v,w\rangle_{\partial \Omega} = \int_{\partial \Omega} v\cdot w~ds$$

We divide the boundary into $\partial \Omega=\Gamma_D \cup \Gamma_N$, with boundary conditions

$$u = g_D,\quad x\in \Gamma_D,$$
$$-\nu \nabla u\cdot n = g_N, \quad x\in \Gamma_N.$$

For $x\in \Gamma_D$ the test function $v=0$.  With $g_N=0$ the boundary term vanishes.

The equations are expressed in residual form

$$r(u;v) = (\nabla u,\nabla v) - (f,v) = 0.\,$$\

# **Method**

**Define domain and mesh**

In [ ]:
# Domain and problem parameters
L = 4
H = 2
resolution = 32

circles = [
    (1.5, 0.25 * H, 0.2),
    (0.5, 0.50 * H, 0.2),
    (2.0, 0.75 * H, 0.2),
]

# Generate mesh: rectangle minus circular holes
msh, cell_tags = gmsh_rect_minus_circles(L, H, circles, resolution)

# Local mesh refinement around (1.5, 0.5) — default no_levels=0
no_levels = 0
for _ in range(no_levels):
    msh, _, _ = refine_cells(
        msh,
        lambda pts: np.sqrt((pts[:, 0] - 1.5)**2 + (pts[:, 1] - 0.5)**2) < 1.0,
    )
facet_tags = tag_boundaries(msh, L, H)

plot_mesh(msh, title=f"Mesh: {msh.topology.index_map(msh.topology.dim).size_global} cells")

**Define finite element approximation space**

In [ ]:
V = functionspace(msh, ("Lagrange", 1))
u = Function(V)
v = ufl.TestFunction(V)

**Define boundary conditions**

Boundary identification uses `tag_boundaries(msh, L, H)` which geometrically
assigns facet tags on the current mesh: left=1, right=2, lower=3, upper=4,
circle boundaries=5.  Tags are recomputed after refinement so they remain valid.
This replaces the indicator `Expression`s from the legacy FEniCS version — `ds(tag)` takes
the role of `ib*ds`, `wb*ds`, and `ob*ds`.

In [ ]:
uin  = 1.0   # inflow value  (left boundary, tag 1)
uout = 0.0   # outflow value (right boundary, tag 2)
uw   = 0.0   # wall value    (lower=3, upper=4, objects=5)

# **Results**

**Define and solve variational problem**

In [ ]:
dx = ufl.Measure("dx", domain=msh)
ds = ufl.Measure("ds", domain=msh, subdomain_data=facet_tags)
h  = ufl.CellDiameter(msh)
x  = ufl.SpatialCoordinate(msh)

C     = 1.0e3
gamma = C / h

f = 10.0 * ufl.sin(x[0])

# Residual form with weak penalty BCs
residual = (
    ufl.inner(ufl.grad(u), ufl.grad(v)) * dx
    + gamma * ufl.inner(u - uin,  v) * ds(1)   # inflow (left)
    + gamma * ufl.inner(u - uout, v) * ds(2)   # outflow (right)
    + gamma * ufl.inner(u - uw,   v) * ds(3)   # wall (lower)
    + gamma * ufl.inner(u - uw,   v) * ds(4)   # wall (upper)
    + gamma * ufl.inner(u - uw,   v) * ds(5)   # wall (objects)
    - ufl.inner(f, v) * dx
)

problem = NonlinearProblem(
    residual, u,
    petsc_options_prefix="poisson_",
    petsc_options={
        "snes_type": "newtonls",
        "snes_rtol": 1.0e-8,
        "snes_atol": 1.0e-10,
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
    },
)
u = problem.solve()
assert problem.solver.getConvergedReason() > 0, (
    f"Newton did not converge: reason={problem.solver.getConvergedReason()}"
)
print(f"Newton converged in {problem.solver.getIterationNumber()} iteration(s).")

**Visualize solution and export files**

In [ ]:
plot_p1(u, title="Solution to Poisson equation")
export_xdmf("results-Poisson/u.xdmf", [u])
# On Colab, pass download=True to trigger a browser download:
# export_xdmf("results-Poisson/u.xdmf", [u], download=True)

In [ ]:
tdim = msh.topology.dim
L2   = assemble_scalar(form(ufl.inner(u, u) * dx))**0.5
intg = assemble_scalar(form(u * dx))

print(f"‖u‖_L²   = {L2:.6f}")
print(f"∫u dx    = {intg:.6f}")
print(f"min u    = {u.x.array.min():.6f}")
print(f"max u    = {u.x.array.max():.6f}")
print(f"cells    = {msh.topology.index_map(tdim).size_global}")
print(f"dofs     = {V.dofmap.index_map.size_global * V.dofmap.index_map_bs}")

# **Discussion**

A finite element method was implemented in FEniCSx to solve the Poisson equation in 2D.
The method was tested for the model problem of flow past a number of circular obstacles,
and the solution behaved as expected.